# CS 3110/5110: Data Privacy

## Homework 5


In [9]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt


def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)


def laplace_mech_vec(qs, sensitivity, epsilon):
    return [laplace_mech(q, sensitivity, epsilon) for q in qs]


def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(
        loc=0, scale=sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    )


def gaussian_mech_vec(qs, sensitivity, epsilon, delta):
    return [gaussian_mech(q, sensitivity, epsilon, delta) for q in qs]


def pct_error(orig, priv):
    return np.abs(orig - priv) / orig * 100.0


adult = pd.read_csv(
    "https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv"
)

## Question 1 (5 points)

Implement the `dp_occupation_histogram` function below. It should return a differentially private histogram over the `Occupation` column in the `adult` dataset. Your function should have a total privacy cost of `epsilon` and should use parallel composition.


In [10]:
def dp_occupation_histogram(epsilon):
    # We grab the "Occupation" column of the adult dataset,
    # then we use "value_counts()" to turn it into a histogram.
    hist = adult["Occupation"].value_counts()

    # Because the different columns are getting the laplace mechanism applied
    # individually, we can ustilize parallel composition to reduce the epsilon
    # cost of making the histogram Differentially Private.
    noisy_hist = hist.apply(
        lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon)
    )
    return noisy_hist


dp_occupation_histogram(1.0)

Occupation
Prof-specialty       4139.757439
Craft-repair         4099.592906
Exec-managerial      4066.411404
Adm-clerical         3770.916648
Sales                3650.783576
Other-service        3296.658912
Machine-op-inspct    2001.303357
Transport-moving     1598.678388
Handlers-cleaners    1371.709914
Farming-fishing       994.775854
Tech-support          928.452219
Protective-serv       651.337102
Priv-house-serv       147.663976
Armed-Forces           11.013246
Name: count, dtype: float64

In [11]:
# TEST CASE
hist = adult["Occupation"].value_counts()
dp_results = [np.abs(dp_occupation_histogram(1.0) - hist) for _ in range(200)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1 / 1.0) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 2
assert stats.wasserstein_distance(all_dp_results, spec) > 0

dp_results = [np.abs(dp_occupation_histogram(0.01) - hist) for _ in range(200)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1 / 0.01) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 150
assert stats.wasserstein_distance(all_dp_results, spec) > 50

## Question 2 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your solution for `dp_occupation_histogram` satisfies `epsilon`-differential privacy.
- How would your results change (in terms of accuracy) if you used sequential composition instead of parallel composition to implement `dp_occupation_histogram`, and why?
- If you partition the data by _both_ occupation and age (i.e. a contingency table), would parallel composition still apply? Why or why not?
- If you partition the data by both occupation and age, what would happen to accuracy, and why?


- 1. Sensitivity: The entries of this histogram are all counting queries, thus
     each sensitivity value is 1.
  2. Composition: This histogram is a set of disjoint subsets of the original
     dataset, therefore parallel composition applies. The total privacy cost is
     therefore epsilon.
  3. Post-Processing: All elements of the returned histogram have had noise
     added to them, thus satisfying differential privacy.
- If I used sequential composition instead of parallel (while still maintaining
  the same total epsilon value) then I would have to reduce the epsilon values
  used by each laplace mechanism. Reducing the epsilon values would result in
  greater noise being introduced, which would in turn reduce the utility of the
  sequential composition version.
- Yes. The resulting histogram partitions would still be disjoint subsets of the
  original dataset; each dataset entry is only counted in a single location in
  the resulting histogram. Therefore we could still apply parallel composition.
- The resulting accuracy would be lower. As we partition the same dataset into
  more refined disjoint subsets, their count values would be smaller. Smaller
  count values, with the same Laplace noise scales, would result in relatively
  larger noise.


## Question 3 (5 points)

Implement the `dp_3d_histogram` function below. It should return a differentially private 3-dimensional histogram over the `Occupation`, `Education`, and `Sex` columns (in that order) in the `adult` dataset (i.e. one count for each combination of occupation, education, and sex). As in Question 1, your function should have a total privacy cost of `epsilon` and should use parallel composition.

_Hint_: `value_counts` works on multiple columns.


In [12]:
def dp_3d_histogram(epsilon):
    hist = adult[["Occupation", "Education", "Sex"]].value_counts()

    # Apply the Laplace Mechanism to each count in the histogram to make it differentially private.
    noisy_hist = hist.apply(
        lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon)
    )

    # Return the, now, differentially private 3D histogram.
    return noisy_hist


dp_3d_histogram(1.0)

Occupation         Education     Sex   
Craft-repair       HS-grad       Male      1820.736506
Exec-managerial    Bachelors     Male      1037.251607
Adm-clerical       HS-grad       Female     966.987809
Prof-specialty     Bachelors     Male       916.697734
Adm-clerical       Some-college  Female     903.991471
                                              ...     
Transport-moving   Doctorate     Male         0.724362
Protective-serv    9th           Female       1.303606
Other-service      Prof-school   Female       0.487561
Handlers-cleaners  1st-4th       Female       1.746638
Machine-op-inspct  Doctorate     Female       2.636917
Name: count, Length: 361, dtype: float64

In [13]:
# TEST CASE
hist = adult[["Occupation", "Education", "Sex"]].value_counts()
dp_results = [(dp_3d_histogram(1.0) - hist).dropna() for _ in range(50)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1 / 1.0) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 0.3
assert stats.wasserstein_distance(all_dp_results, spec) > 0

dp_results = [(dp_3d_histogram(0.1) - hist).dropna() for _ in range(50)]
all_dp_results = [item for sublist in dp_results for item in sublist]
spec = [np.random.laplace(loc=0, scale=1 / 0.1) for _ in range(2000)]
assert stats.wasserstein_distance(all_dp_results, spec) < 3
assert stats.wasserstein_distance(all_dp_results, spec) > 0

## Question 4 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your solution for `dp_3d_histogram` satisfies `epsilon`-differential privacy.
- Which histogram do you expect to have higher relative error (percent error): the 1-dimensional one from Question 1, or the 3-dimensional one from Question 3? Why?


- 1. Sensitivity: The entries of this histogram are all counting queries, thus
     each sensitivity value is 1.
  2. Composition: This 3D histogram is, still, a set of disjoint subsets of the original
     dataset, therefore parallel composition applies. The total privacy cost is
     therefore epsilon.
  3. Post-Processing: All elements of the returned histogram have had noise
     added to them, thus satisfying differential privacy.
- The 3-dimensional one from Question 3 would have higher relative error. The 3D
  histogram has smaller count entries than the 1D histogram, but the scale of
  the noise applied to both are the same. Because the 3D counts are lower then,
  the relative errors would be higher.


## Question 5 (5 points)

Implement the `dp_three_histograms` function below. It should return a tuple of three differentially private histograms:

1. A 1-dimensional histogram over the `Occupation` column
2. A 2-dimensional histogram over the `Occupation` and `Education` columns
3. A 3-dimensional histogram over the `Occupation`, `Education`, and `Sex` columns

Your function should have a total privacy cost of `epsilon`.

Hint: take care with composition!


In [14]:
def dp_three_histograms(epsilon):
    hist_1 = adult["Occupation"].value_counts()
    hist_2 = adult[["Occupation", "Education"]].value_counts()
    hist_3 = adult[["Occupation", "Education", "Sex"]].value_counts()

    # Apply the Laplace mechanism to each histogram to make them differentially
    # private. Take care to split up the epsilon budget appropriately among the
    # histograms.
    noisy_hist_1 = hist_1.apply(
        lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon / 3)
    )
    noisy_hist_2 = hist_2.apply(
        lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon / 3)
    )
    noisy_hist_3 = hist_3.apply(
        lambda x: laplace_mech(x, sensitivity=1, epsilon=epsilon / 3)
    )

    # Returning all three histograms as a tuple is a form of sequential
    # composition, akin to adding the results of each differentially private
    # query together. Therefore the total privacy cost here is the summation of
    # the three privacy costs used for each histogram.
    return noisy_hist_1, noisy_hist_2, noisy_hist_3


dp_three_histograms(1.0)

(Occupation
 Prof-specialty       4139.568351
 Craft-repair         4099.812739
 Exec-managerial      4066.792999
 Adm-clerical         3770.580815
 Sales                3651.957033
 Other-service        3299.299648
 Machine-op-inspct    2000.942671
 Transport-moving     1600.857311
 Handlers-cleaners    1367.319938
 Farming-fishing       996.307626
 Tech-support          926.034495
 Protective-serv       642.067310
 Priv-house-serv       149.622337
 Armed-Forces           14.562765
 Name: count, dtype: float64,
 Occupation         Education
 Craft-repair       HS-grad      1919.221467
 Prof-specialty     Bachelors    1498.982809
 Exec-managerial    Bachelors    1375.670324
 Adm-clerical       HS-grad      1371.706562
 Other-service      HS-grad      1281.988300
                                    ...     
 Prof-specialty     Preschool       0.799487
 Armed-Forces       Masters         1.868624
 Tech-support       5th-6th         1.330251
 Transport-moving   Doctorate       5.769793
 M

In [15]:
# TEST CASE
hists = [
    adult["Occupation"].value_counts(),
    adult[["Occupation", "Education"]].value_counts(),
    adult[["Occupation", "Education", "Sex"]].value_counts(),
]
dp_results = [dp_three_histograms(1.0) for _ in range(100)]
assert all(len(r) == 3 for r in dp_results)

for i, hist in enumerate(hists):
    all_dp_results = [
        item for r in dp_results for item in (r[i] - hist).dropna()
    ]
    spec = [np.random.laplace(loc=0, scale=3 / 1.0) for _ in range(2000)]
    assert stats.wasserstein_distance(all_dp_results, spec) < 1
    assert stats.wasserstein_distance(all_dp_results, spec) > 0

## Question 6 (5 points)

Using the 3-point structure described in lecture, argue that your solution for `dp_three_histograms` satisfies `epsilon`-differential privacy. Be precise about composition.


1. Sensitivity: The entries of each histogram are all counting queries, thus each sensitivity value is 1.
2. Composition: The privacy cost for each of the three _individual_ histograms is
   `epsilon/3`, by parallel composition. We are able to use parallel composition
   for each histogram because the histogram entries are disjoint subsets. The _total_ privacy cost for returning
   all three of these histograms is `3 * epsilon/3 = epsilon`, by sequential
   composition. We utilize parallel composition within each histogram, but can
   only use sequential composition when returning all three because the
   histograms are not disjoint from each other, only within themselves.
3. Post-Processing: All three returned histograms have had noise added to them via the
   Laplace Mechanism and therefore satisy DP.
